# Notebook 2 — National Daymet Climate Features

Computes annual climate features for every U.S. bridge by pulling daily Daymet
data per unique ~5.5 km grid cell, engineering the same indicators as the MA
pipeline (`../ma_bridges/bridge_project.ipynb`) plus national-scale additions (heat, humidity,
solar, scour proxies).

**Method:** per-point `pydaymet.get_bycoords`, parallelised + cached + resumable
(mirrors `download_nbi_national.ipynb`). Cost scales with unique cells (~159K at
0.05°), not the ~1.21M bridges. Climate is keyed by **cell**, with a small
bridge→cell map; the per-bridge-year expansion happens at model-build time.

**Outputs (in `Bridges_all_of_US/`):**
- `us_bridge_cell_map.csv` — STRUCTURE_NUMBER_008 → grid cell
- `us_climate_by_cell_year.csv` — annual features per (cell, year)

**Notes baked in from live testing:** `get_bycoords` returns a DataFrame with a
`time` index and `(unit)`-suffixed columns; it **ignores the `dates` arg** and
returns 1980→2025, so we filter to 1992–2025 after fetching (2025 is available).

In [ ]:
import os, time, glob, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import pydaymet as daymet

# pyproj raises a NumPy-2 DeprecationWarning ("Conversion of an array with ndim > 0
# to a scalar ...") on EVERY coordinate transform. Jupyter re-enables DeprecationWarning
# display, so it floods the output from the 24 worker threads. Silence it globally and
# early here; fetch_cell also wraps the call in catch_warnings as a per-thread backstop.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message="Conversion of an array")

print("pydaymet", daymet.__version__)

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
START_YEAR  = 1992
END_YEAR    = 2025            # Daymet already publishes 2025 in this environment
GRID_DEG    = 0.05           # ~5.5 km snapping (chosen resolution)
DAYMET_VARS = ["tmax", "tmin", "prcp", "swe", "vp", "srad"]
MAX_WORKERS = 24            # parallel cell fetches (~5.6 s/cell). 24 only works on a
                            # DEEP queue; a small residual synchronises its retries and
                            # trips ORNL's rate limiter. Failed cells are re-fetched on
                            # re-run (resumable), so this is safe to tune.
BATCH_SIZE  = 500           # cells per checkpoint CSV
RETRIES     = 3

RAW_DIR      = Path("raw")
CACHE_DIR    = Path("climate_cache"); CACHE_DIR.mkdir(exist_ok=True)
CELL_MAP_OUT = Path("us_bridge_cell_map.csv")
CLIMATE_OUT  = Path("us_climate_by_cell_year.csv")
LOG_FILE     = Path("climate_download_log.txt")
print("config ready")

In [ ]:
# ── Build the unique-cell list and bridge->cell map (valid-first + county imputation) ──
# Canonical NBI packed-DMS -> decimal conversion (from ../ma_bridges/bridge_map.ipynb).
def nbi_to_decimal(val):
    degrees = val // 1000000
    minutes = (val % 1000000) // 10000
    seconds = (val % 10000) / 100
    return degrees + minutes / 60 + seconds / 3600

def load_unique_bridges():
    # Memory-safe: stream each per-state file in chunks and reduce immediately (a plain
    # full-file read OOMs on ~20M rows once the kernel already holds the climate table).
    #
    # VALID-COORD-FIRST: filter each chunk to rows with a good in-bounds coordinate BEFORE
    # deduping, so a bridge whose FIRST survey row lacks a coord is still captured from a
    # later good row (the old dedup-then-filter dropped ~130K such bridges, mostly IA/MS
    # whose early surveys had no lat/lon -> ~18% of the Notebook 4 panel had no cell match).
    valid_parts, all_parts = [], []
    for f in sorted(RAW_DIR.glob("*_bridges.csv")):
        vp, ap = [], []
        for chunk in pd.read_csv(
                f, usecols=["STRUCTURE_NUMBER_008", "STATE_FIPS", "COUNTY_CODE_003", "LAT_016", "LONG_017"],
                dtype={"STRUCTURE_NUMBER_008": str, "STATE_FIPS": str, "COUNTY_CODE_003": str}, chunksize=200_000):
            st = chunk["STATE_FIPS"].str.strip(); sn = chunk["STRUCTURE_NUMBER_008"].str.strip()
            co = chunk["COUNTY_CODE_003"].str.strip()
            lat_raw = pd.to_numeric(chunk["LAT_016"], errors="coerce")
            lon_raw = pd.to_numeric(chunk["LONG_017"], errors="coerce")
            lat = nbi_to_decimal(lat_raw); lon = -nbi_to_decimal(lon_raw)   # West = negative
            v = lat_raw.gt(0) & lon_raw.gt(0) & lat.between(15, 72) & lon.between(-170, -60)
            ap.append(pd.DataFrame({"STATE_FIPS": st, "STRUCTURE_NUMBER_008": sn, "COUNTY_CODE_003": co}))
            vp.append(pd.DataFrame({"STATE_FIPS": st[v], "STRUCTURE_NUMBER_008": sn[v],
                                    "COUNTY_CODE_003": co[v], "lat": lat[v], "lon": lon[v]}))
        valid_parts.append(pd.concat(vp, ignore_index=True).drop_duplicates(["STATE_FIPS","STRUCTURE_NUMBER_008"]))
        all_parts.append(pd.concat(ap, ignore_index=True).drop_duplicates(["STATE_FIPS","STRUCTURE_NUMBER_008"]))
    valb = pd.concat(valid_parts, ignore_index=True).drop_duplicates(["STATE_FIPS","STRUCTURE_NUMBER_008"])
    allb = pd.concat(all_parts,   ignore_index=True).drop_duplicates(["STATE_FIPS","STRUCTURE_NUMBER_008"])

    # COUNTY-CENTROID IMPUTATION: ~8% of bridges have NO valid coordinate in ANY survey year
    # (genuinely blank lat/lon, not a format issue). Give each its county centroid (state
    # centroid fallback) so it still maps to a cell + gets coastal distance; flag it.
    cty = valb.groupby(["STATE_FIPS","COUNTY_CODE_003"])[["lat","lon"]].mean().reset_index()
    sta = valb.groupby(["STATE_FIPS"])[["lat","lon"]].mean().reset_index().rename(columns={"lat":"slat","lon":"slon"})
    vkeys = set(map(tuple, valb[["STATE_FIPS","STRUCTURE_NUMBER_008"]].values))
    cl = allb[~allb.set_index(["STATE_FIPS","STRUCTURE_NUMBER_008"]).index.isin(vkeys)].copy()
    cl = cl.merge(cty, on=["STATE_FIPS","COUNTY_CODE_003"], how="left").merge(sta, on="STATE_FIPS", how="left")
    cl["lat"] = cl["lat"].fillna(cl["slat"]); cl["lon"] = cl["lon"].fillna(cl["slon"])
    cl = cl.dropna(subset=["lat","lon"])

    valb["location_imputed"] = np.int8(0); cl["location_imputed"] = np.int8(1)
    out = pd.concat([valb, cl[valb.columns]], ignore_index=True)
    out["grid_lat"] = np.round(np.round(out["lat"] / GRID_DEG) * GRID_DEG, 3)
    out["grid_lon"] = np.round(np.round(out["lon"] / GRID_DEG) * GRID_DEG, 3)
    # dedup on (STATE_FIPS, STRUCTURE_NUMBER_008); structure # is unique only WITHIN a state.
    return out.drop_duplicates(["STATE_FIPS","STRUCTURE_NUMBER_008"])

bridges = load_unique_bridges()
bridge_cell_map = bridges[["STRUCTURE_NUMBER_008", "STATE_FIPS", "grid_lat", "grid_lon", "location_imputed"]]
bridge_cell_map.to_csv(CELL_MAP_OUT, index=False)

# The download only fetches MEASURED cells. The extra cells that imputed bridges reference are
# NOT auto-fetched (keeps the run bounded) -> those imputed bridges' climate stays NaN unless
# you opt into the top-up: swap `== 0` for `.notna()` below to also fetch the imputed cells.
unique_cells = (bridge_cell_map.loc[bridge_cell_map["location_imputed"] == 0, ["grid_lat", "grid_lon"]]
                .drop_duplicates().reset_index(drop=True))
print(f"{len(bridges):,} bridges ({int(bridges['location_imputed'].sum()):,} county-imputed) "
      f"-> {len(unique_cells):,} measured {GRID_DEG} deg cells to fetch")
print(f"bridge->cell map saved to {CELL_MAP_OUT}")

In [ ]:
# ── Feature engineering: daily series (one cell) -> annual features ───────────
# Originals match bridge_project.ipynb verbatim; the rest are the national adds.
def compute_annual_features(d):
    d["freeze_thaw_day"]   = (d.tmin < 0) & (d.tmax > 0)
    d["freeze_degree_day"] = np.where(d.tmin < 0, -d.tmin, 0)
    d["snow_day"]          = (d.prcp > 0) & (d.tmax < 0)
    d["hot_day"]           = d.tmax > 32          # >89.6 F
    d["heavy_rain"]        = d.prcp > 25          # mm/day
    d["wet_day"]           = d.prcp > 1
    d["diurnal"]           = d.tmax - d.tmin
    ann = d.groupby("year").agg(
        freeze_thaw_cycles=("freeze_thaw_day", "sum"),
        freezing_index=("freeze_degree_day", "sum"),
        annual_precip_mm=("prcp", "sum"),
        snow_days=("snow_day", "sum"),
        max_swe=("swe", "max"),
        hot_days=("hot_day", "sum"),
        annual_max_tmax=("tmax", "max"),
        heavy_rain_days=("heavy_rain", "sum"),
        wet_days=("wet_day", "sum"),
        max_1day_precip=("prcp", "max"),
        mean_vp=("vp", "mean"),
        mean_srad=("srad", "mean"),
        mean_diurnal_range=("diurnal", "mean"),
    )
    d["season"] = d.month.map({12: "DJF", 1: "DJF", 2: "DJF", 6: "JJA", 7: "JJA", 8: "JJA"})
    seas = (d.dropna(subset=["season"]).groupby(["year", "season"])
              .agg(mean_tmax=("tmax", "mean"), mean_tmin=("tmin", "mean")).unstack("season"))
    seas.columns = [f"{a}_{b}" for a, b in seas.columns]
    return ann.join(seas).reset_index()

def standardize(df):
    # map "tmin (degrees C)" -> "tmin", etc.
    rename = {}
    for c in df.columns:
        base = c.split("(")[0].strip().lower()
        if base in ["tmax", "tmin", "prcp", "swe", "srad", "vp", "dayl"]:
            rename[c] = base
    df = df.rename(columns=rename).reset_index().rename(columns={"time": "date"})
    df["date"] = pd.to_datetime(df["date"])
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    return df

def region_for(grid_lat, grid_lon):
    # Daymet ships separate tiles. 'na' (continental) covers lon[-136.9, -6.1],
    # lat[6.08, 69.08]; Hawaii is its own 'hi' tile. Anything west of -136.9
    # (interior/N/W Alaska) has NO Daymet coverage in any tile.
    if 17.5 <= grid_lat <= 23.5 and -161.0 <= grid_lon <= -154.0:
        return "hi"
    return "na"

def fetch_cell(grid_lat, grid_lon):
    region = region_for(grid_lat, grid_lon)
    # srad (solar radiation) is no-data at many coastal-edge Daymet pixels, and a
    # single unavailable variable makes get_bycoords 400 the WHOLE request. Try the
    # full 6-var set first (so inland cells keep srad), then fall back to dropping
    # srad -> mean_srad becomes NaN for that cell only. If even the reduced set
    # fails, the pixel is true no-data (over water) -> RETRY.
    var_sets = (DAYMET_VARS, [v for v in DAYMET_VARS if v != "srad"])
    last_err = None
    for attempt in range(RETRIES):
        for variables in var_sets:
            try:
                # catch_warnings backstop: silence pyproj's per-transform DeprecationWarning
                # right at the worker-thread call site (Jupyter can re-enable it globally).
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    raw = daymet.get_bycoords((grid_lon, grid_lat),
                                              dates=(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"),
                                              variables=variables, region=region)
                d = standardize(raw)
                d = d[(d.year >= START_YEAR) & (d.year <= END_YEAR)]
                if "srad" not in d.columns:
                    d["srad"] = np.nan            # keep mean_srad column, filled NaN
                ann = compute_annual_features(d)
                ann["grid_lat"] = grid_lat
                ann["grid_lon"] = grid_lon
                return ann, None
            except Exception as e:
                msg = str(e)
                # Out-of-domain coords (e.g. interior/N/W Alaska) raise InputRangeError
                # and will NEVER succeed -> flag permanent so the driver dead-lists the
                # cell (no retry, no re-attempt on later runs).
                if type(e).__name__ == "InputRangeError" or "Valid range for coords" in msg:
                    return None, ("DEAD", msg)
                last_err = msg
        # both var sets failed this attempt -> jittered backoff, then retry. This
        # de-synchronises the startup worker burst that trips ORNL's rate limiter.
        time.sleep(1.5 * (attempt + 1) + np.random.uniform(0, 2.0))
    return None, ("RETRY", last_err)

print("feature functions ready")

In [ ]:
# ── Parallel, resumable download driver ──────────────────────────────────────
# Re-running is safe: cells already in climate_cache/*.csv are skipped, and cells
# with no Daymet coverage (interior/N/W Alaska) are recorded in dead_cells.csv so
# they are never re-attempted on later runs.
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

DEAD_FILE = CACHE_DIR / "dead_cells.csv"

def completed_cells():
    done = set()
    for p in CACHE_DIR.glob("part_*.csv"):
        try:
            t = pd.read_csv(p, usecols=["grid_lat", "grid_lon"])
            done.update(map(tuple, t[["grid_lat", "grid_lon"]].drop_duplicates().values))
        except Exception:
            pass
    return done

def load_dead():
    if DEAD_FILE.exists():
        try:
            d = pd.read_csv(DEAD_FILE)
            return set(map(tuple, d[["grid_lat", "grid_lon"]].values))
        except Exception:
            pass
    return set()

done = completed_cells()
dead = load_dead()
skip = done | dead
todo = [tuple(r) for r in unique_cells[["grid_lat", "grid_lon"]].values if tuple(r) not in skip]
print(f"{len(done):,} cached, {len(dead):,} dead (no Daymet); {len(todo):,} to fetch.")

def flush(batch, idx):
    if batch:
        pd.concat(batch, ignore_index=True).to_csv(CACHE_DIR / f"part_{idx:05d}.csv", index=False)

# dead cells are appended immediately (and flushed) so an interrupt keeps them
if not DEAD_FILE.exists():
    pd.DataFrame(columns=["grid_lat", "grid_lon"]).to_csv(DEAD_FILE, index=False)
dead_fh = open(DEAD_FILE, "a")

errors = []
batch = []
part_idx = len(list(CACHE_DIR.glob("part_*.csv")))
t0 = time.time(); n = 0; n_dead = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futs = {pool.submit(fetch_cell, la, lo): (la, lo) for la, lo in todo}
    for fut in as_completed(futs):
        la, lo = futs[fut]
        ann, err = fut.result(); n += 1
        if err:
            kind, msg = err
            if kind == "DEAD":
                dead_fh.write(f"{la},{lo}\n"); dead_fh.flush(); n_dead += 1
            else:
                errors.append(f"{la},{lo}: {msg}")
        elif ann is not None:
            batch.append(ann)
            if len(batch) >= BATCH_SIZE:
                flush(batch, part_idx); part_idx += 1; batch = []
        if n % 500 == 0:
            print(f"  {n:,}/{len(todo):,}  ({(time.time()-t0)/60:.1f} min, "
                  f"{len(errors)} transient errors, {n_dead} dead)")

flush(batch, part_idx)
dead_fh.close()
with open(LOG_FILE, "w") as f:
    f.write("\n".join(errors) if errors else "No transient errors.")
print(f"Done. {n:,} processed, {len(errors)} transient errors, {n_dead} dead-listed this run -> {LOG_FILE}")

In [ ]:
# ── Assemble the per-cell-year climate table ─────────────────────────────────
parts = [pd.read_csv(p) for p in sorted(CACHE_DIR.glob("part_*.csv"))]
climate = pd.concat(parts, ignore_index=True)
climate = climate.rename(columns={"year": "SURVEY_YEAR"})
climate = climate.drop_duplicates(["grid_lat", "grid_lon", "SURVEY_YEAR"])
climate.to_csv(CLIMATE_OUT, index=False)
print(f"{len(climate):,} cell-year rows, {climate[['grid_lat','grid_lon']].drop_duplicates().shape[0]:,} cells -> {CLIMATE_OUT}")
print(climate.columns.tolist())

In [ ]:
# ── Sanity checks ────────────────────────────────────────────────────────────
print(climate[["freeze_thaw_cycles", "hot_days", "annual_precip_mm", "max_swe",
               "mean_tmax_JJA", "mean_tmin_DJF"]].describe().round(1).to_string())
print("\nNaN per column:")
print(climate.isna().sum().to_string())